# w_loss_12_345 Loss Weight Sweep - 1-layer MLP

Binary labels:
- 0: confidence 1, 2
- 1: confidence 3, 4, 5

This notebook keeps the original train distribution and sweeps loss weights for original confidence 1 and 2 samples.

Model structure is the same as the previous up_sampling_confi4 1-layer MLP: `Dropout -> Linear`.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
TEST_SIZE = 0.20
VALID_SIZE = 0.20
BATCH_SIZE = 256
NUM_EPOCHS = 120
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5
ORIGINAL_LABELS = [1, 2, 3, 4, 5]
BINARY_LABELS = [0, 1]
NEGATIVE_CONFIDENCES = [1, 2]
POSITIVE_CONFIDENCES = [3, 4, 5]
BINARY_LABEL_NAMES = {0: 'confidence12', 1: 'confidence345'}
EXPERIMENT_NAME = 'w_loss_12_345_loss_weight_sweep_1layer_mlp_binary_classifier'

LOSS_WEIGHT_SWEEP = [
    {'tag': 'baseline_w1_1_w2_1', 'w1': 1.0, 'w2': 1.0},
    {'tag': 'mild_w1_5_w2_1p5', 'w1': 5.0, 'w2': 1.5},
    {'tag': 'medium_w1_10_w2_2', 'w1': 10.0, 'w2': 2.0},
    {'tag': 'current_w1_20p54_w2_2p93', 'w1': 20.540625, 'w2': 2.927839643652561},
    {'tag': 'strong_w1_30_w2_4', 'w1': 30.0, 'w2': 4.0},
    {'tag': 'very_strong_w1_40_w2_6', 'w1': 40.0, 'w2': 6.0},
]

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
print('Binary task:', BINARY_LABEL_NAMES[0], 'vs', BINARY_LABEL_NAMES[1])

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'metadata' / 'BSD10k_metadata.csv').is_file():
            return candidate
    raise FileNotFoundError('Could not find data/metadata/BSD10k_metadata.csv')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
FEATURE_DIR = DATA_DIR / 'features'
METADATA_CSV = DATA_DIR / 'metadata' / 'BSD10k_metadata.csv'
AUDIO_EMB_DIR = FEATURE_DIR / 'clap_audio_embeddings'
TEXT_EMB_DIR = FEATURE_DIR / 'clap_text_embeddings'
OUTPUT_BASE_DIR = PROJECT_ROOT / 'outputs' / 'up_sampling_confi4_binary_w_loss' / EXPERIMENT_NAME
PLOT_DIR = OUTPUT_BASE_DIR / 'plots'
REPORT_DIR = OUTPUT_BASE_DIR / 'reports'
PRED_DIR = OUTPUT_BASE_DIR / 'predictions'
MODEL_DIR = OUTPUT_BASE_DIR / 'models'
for p in [PLOT_DIR, REPORT_DIR, PRED_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_BASE_DIR:', OUTPUT_BASE_DIR)

In [ ]:
def finite_float32(x):
    return np.nan_to_num(np.asarray(x, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def clean_metadata(df):
    df = df.copy()
    df['sound_id'] = df['sound_id'].astype(str).str.strip()
    df['class'] = df['class'].astype(str).str.strip()
    df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')
    df = df[df['confidence'].isin(ORIGINAL_LABELS)].copy()
    df['confidence'] = df['confidence'].astype(int)
    if 'class_idx' in df.columns:
        class_idx = df['class_idx'].astype(str).str.strip()
        keep = ~((class_idx.str.len() == 3) & (class_idx.str.endswith('99') | class_idx.str.endswith('00')))
        df = df[keep].copy()
    return df.reset_index(drop=True)

def one_hot(values, categories):
    category_to_idx = {category: idx for idx, category in enumerate(categories)}
    arr = np.zeros((len(values), len(categories)), dtype=np.float32)
    for row_idx, value in enumerate(values):
        arr[row_idx, category_to_idx[str(value)]] = 1.0
    return arr

def load_embedding(path):
    return finite_float32(np.load(path).reshape(-1))

def make_binary_label(conf):
    if int(conf) in NEGATIVE_CONFIDENCES:
        return 0
    if int(conf) in POSITIVE_CONFIDENCES:
        return 1
    raise ValueError(conf)

def build_dataset():
    meta = clean_metadata(pd.read_csv(METADATA_CSV))
    audio_paths = {p.stem: p for p in AUDIO_EMB_DIR.glob('*.npy')}
    text_paths = {p.stem: p for p in TEXT_EMB_DIR.glob('*.npy')}
    rows, features = [], []
    class_categories = sorted(meta['class'].astype(str).unique().tolist())
    class_oh = one_hot(meta['class'].astype(str).tolist(), class_categories)
    for row_idx, row in meta.iterrows():
        sid = str(row['sound_id'])
        if sid not in audio_paths or sid not in text_paths:
            continue
        x = np.concatenate([load_embedding(audio_paths[sid]), load_embedding(text_paths[sid]), class_oh[row_idx]], axis=0).astype(np.float32)
        rows.append({'sound_id': sid, 'class': row['class'], 'confidence': int(row['confidence']), 'binary_label': make_binary_label(row['confidence'])})
        features.append(x)
    return pd.DataFrame(rows), np.stack(features).astype(np.float32), class_categories

df, X, class_categories = build_dataset()
y_conf = df['confidence'].to_numpy(dtype=np.int64)
y_bin = df['binary_label'].to_numpy(dtype=np.int64)
train_val_idx, test_idx = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED, stratify=y_conf)
train_idx, valid_idx = train_test_split(train_val_idx, test_size=VALID_SIZE, random_state=SEED, stratify=y_conf[train_val_idx])

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
X_valid = scaler.transform(X[valid_idx]).astype(np.float32)
X_test = scaler.transform(X[test_idx]).astype(np.float32)
y_train, y_valid, y_test = y_bin[train_idx], y_bin[valid_idx], y_bin[test_idx]
y_train_conf, y_valid_conf, y_test_conf = y_conf[train_idx], y_conf[valid_idx], y_conf[test_idx]

df.to_csv(REPORT_DIR / 'dataset_index.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'sound_id': df['sound_id'], 'split': 'train'}).assign(split=lambda s: s['split']).to_csv(REPORT_DIR / 'dataset_index_placeholder.csv', index=False, encoding='utf-8-sig')
print('X:', X.shape)
display(pd.Series(y_conf).value_counts().reindex(ORIGINAL_LABELS, fill_value=0).rename('full_count').to_frame())
display(pd.crosstab(pd.Series(['train'] * len(train_idx) + ['valid'] * len(valid_idx) + ['test'] * len(test_idx)), pd.Series(np.concatenate([y_train_conf, y_valid_conf, y_test_conf])), rownames=['split'], colnames=['confidence']))

In [ ]:
class OneLayerMLPBinaryClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=2, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(input_dim, num_classes))
    def forward(self, x):
        return self.net(x)

def make_loader(X_arr, y_arr, weights=None, shuffle=False):
    if weights is None:
        weights = np.ones(len(y_arr), dtype=np.float32)
    ds = TensorDataset(torch.from_numpy(X_arr), torch.from_numpy(y_arr.astype(np.int64)), torch.from_numpy(weights.astype(np.float32)))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def sample_weights_from_conf(conf_values, w1, w2):
    weight_map = {1: float(w1), 2: float(w2), 3: 1.0, 4: 1.0, 5: 1.0}
    return np.asarray([weight_map[int(c)] for c in conf_values], dtype=np.float32), weight_map

def weighted_cross_entropy(logits, y, sample_weight):
    loss = nn.functional.cross_entropy(logits, y, reduction='none')
    return (loss * sample_weight).sum() / sample_weight.sum().clamp_min(1e-8)

def compute_metrics(y_true, y_pred):
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return {'accuracy': accuracy_score(y_true, y_pred), 'macro_precision': p_macro, 'macro_recall': r_macro, 'macro_f1': f1_macro, 'weighted_precision': p_weighted, 'weighted_recall': r_weighted, 'weighted_f1': f1_weighted, 'mae': mean_absolute_error(y_true, y_pred)}

def train_one(config):
    tag, w1, w2 = config['tag'], config['w1'], config['w2']
    seed_everything(SEED)
    weights, weight_map = sample_weights_from_conf(y_train_conf, w1, w2)
    model = OneLayerMLPBinaryClassifier(X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_loader = make_loader(X_train, y_train, weights, shuffle=True)
    valid_loader = make_loader(X_valid, y_valid, shuffle=False)
    history, best_state, best_val_loss, best_epoch = [], None, float('inf'), 0
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss_sum, train_weight_sum = 0.0, 0.0
        for xb, yb, wb in train_loader:
            xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
            optimizer.zero_grad()
            loss = weighted_cross_entropy(model(xb), yb, wb)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * float(wb.sum().item())
            train_weight_sum += float(wb.sum().item())
        model.eval()
        valid_loss_sum, n_valid, valid_true, valid_pred = 0.0, 0, [], []
        with torch.no_grad():
            for xb, yb, _ in valid_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                valid_loss_sum += nn.functional.cross_entropy(logits, yb, reduction='sum').item()
                n_valid += len(yb)
                valid_true.append(yb.cpu().numpy())
                valid_pred.append(logits.argmax(dim=1).cpu().numpy())
        valid_loss = valid_loss_sum / n_valid
        valid_true = np.concatenate(valid_true)
        valid_pred = np.concatenate(valid_pred)
        row = {'epoch': epoch, 'train_loss': train_loss_sum / max(train_weight_sum, 1e-8), 'valid_loss': valid_loss, **compute_metrics(valid_true, valid_pred)}
        history.append(row)
        if valid_loss < best_val_loss:
            best_val_loss = valid_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    history_df.to_csv(REPORT_DIR / f'{tag}_history.csv', index=False, encoding='utf-8-sig')
    return model, history_df, {'best_epoch': best_epoch, 'best_valid_loss': best_val_loss, 'weight_map': weight_map}

def predict(model, X_arr):
    loader = DataLoader(torch.from_numpy(X_arr), batch_size=BATCH_SIZE, shuffle=False)
    probs = []
    model.eval()
    with torch.no_grad():
        for xb in loader:
            probs.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy())
    prob = np.concatenate(probs)
    return prob.argmax(axis=1), prob

def save_cm(cm, labels, title, path):
    cm_norm = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(5.5, 4.8))
    plt.imshow(cm_norm, vmin=0, vmax=1)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(range(len(labels)), labels)
    plt.yticks(range(len(labels)), labels)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            plt.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    return cm_norm

In [ ]:
results = []
for config in LOSS_WEIGHT_SWEEP:
    tag = config['tag']
    print('\n' + '=' * 80)
    print('Running', tag, 'w1=', config['w1'], 'w2=', config['w2'])
    model, history_df, info = train_one(config)
    test_pred, test_prob = predict(model, X_test)
    metrics = compute_metrics(y_test, test_pred)
    cm = confusion_matrix(y_test, test_pred, labels=BINARY_LABELS)
    cm_norm = save_cm(cm, [BINARY_LABEL_NAMES[i] for i in BINARY_LABELS], f'{tag} row-normalized confusion matrix', PLOT_DIR / f'{tag}_test_confusion_row_normalized.png')
    report_text = classification_report(y_test, test_pred, labels=BINARY_LABELS, target_names=[BINARY_LABEL_NAMES[i] for i in BINARY_LABELS], digits=4, zero_division=0)
    report_dict = classification_report(y_test, test_pred, labels=BINARY_LABELS, target_names=[BINARY_LABEL_NAMES[i] for i in BINARY_LABELS], output_dict=True, zero_division=0)
    (REPORT_DIR / f'{tag}_classification_report.txt').write_text(report_text, encoding='utf-8')
    pd.DataFrame(report_dict).T.to_csv(REPORT_DIR / f'{tag}_classification_report.csv', encoding='utf-8-sig')
    pd.DataFrame(cm, index=[f'true_{BINARY_LABEL_NAMES[i]}' for i in BINARY_LABELS], columns=[f'pred_{BINARY_LABEL_NAMES[i]}' for i in BINARY_LABELS]).to_csv(REPORT_DIR / f'{tag}_test_confusion_counts.csv', encoding='utf-8-sig')
    pd.DataFrame(cm_norm, index=[f'true_{BINARY_LABEL_NAMES[i]}' for i in BINARY_LABELS], columns=[f'pred_{BINARY_LABEL_NAMES[i]}' for i in BINARY_LABELS]).to_csv(REPORT_DIR / f'{tag}_test_confusion_row_normalized.csv', encoding='utf-8-sig')
    pred_out = df.iloc[test_idx][['sound_id', 'class', 'confidence']].copy()
    pred_out['true_binary'] = y_test
    pred_out['pred_binary'] = test_pred
    pred_out['true_binary_name'] = [BINARY_LABEL_NAMES[int(i)] for i in y_test]
    pred_out['pred_binary_name'] = [BINARY_LABEL_NAMES[int(i)] for i in test_pred]
    for i, label in enumerate(BINARY_LABELS):
        pred_out[f'prob_{BINARY_LABEL_NAMES[label]}'] = test_prob[:, i]
    pred_out.to_csv(PRED_DIR / f'{tag}_test_predictions.csv', index=False, encoding='utf-8-sig')
    torch.save({'model_state_dict': model.state_dict(), 'config': config, 'info': info, 'metrics': metrics}, MODEL_DIR / f'{tag}.pt')
    results.append({'tag': tag, 'w1': config['w1'], 'w2': config['w2'], **info, **metrics, 'recall_negative': float(cm_norm[0, 0]), 'recall_positive': float(cm_norm[1, 1])})
    print(report_text)

results_df = pd.DataFrame(results)
results_df.to_csv(REPORT_DIR / 'loss_weight_sweep_summary.csv', index=False, encoding='utf-8-sig')
(REPORT_DIR / 'loss_weight_sweep_summary.json').write_text(json.dumps({'task': '12_vs_345', 'configs': LOSS_WEIGHT_SWEEP, 'results': results}, indent=2, ensure_ascii=False, default=float), encoding='utf-8')
display(results_df.sort_values('macro_f1', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(len(results_df))
ax.plot(x, results_df['macro_f1'], marker='o', label='macro F1')
ax.plot(x, results_df['recall_negative'], marker='o', label=f"recall {BINARY_LABEL_NAMES[0]}")
ax.plot(x, results_df['recall_positive'], marker='o', label=f"recall {BINARY_LABEL_NAMES[1]}")
ax.set_xticks(x)
ax.set_xticklabels(results_df['tag'], rotation=45, ha='right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / 'loss_weight_sweep_tradeoff.png', dpi=180)
plt.show()